In [0]:
from pyspark.sql.functions import concat_ws

# ==========================
# SQL Server Configuration
# ==========================

server_name = "skitjobserver2026.database.windows.net"
database_name = "jobmarketdb"

username = "sqladmin"
password = "KartevSha@2004"

jdbc_port = 1433

jdbc_url = (
    f"jdbc:sqlserver://{server_name}:{jdbc_port};"
    f"database={database_name};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# ==========================
# Test SQL Connection
# ==========================

test_df = (
    spark.read
         .jdbc(
             url=jdbc_url,
             table="(SELECT 1 AS ConnectionSuccessful) AS TestQuery",
             properties=connection_properties
         )
)

display(test_df)

# ==========================
# Read Gold Delta Tables
# ==========================

jobs_fact_df = spark.table("skitkartev.gold.jobs_fact")
skill_demand_df = spark.table("skitkartev.gold.skill_demand")
company_hiring_df = spark.table("skitkartev.gold.company_hiring")
location_hiring_df = spark.table("skitkartev.gold.location_hiring")
salary_summary_df = spark.table("skitkartev.gold.salary_summary")
tech_stack_demand_df = spark.table("skitkartev.gold.tech_stack_demand")

# ==========================
# Convert Array Columns
# ==========================

jobs_fact_sql_df = (
    jobs_fact_df
        .withColumn("RequiredSkills", concat_ws(", ", "RequiredSkills"))
        .withColumn("PreferredSkills", concat_ws(", ", "PreferredSkills"))
)

# ==========================
# Helper Function
# ==========================

def write_to_sql(df, table_name):

    (
        df.write
          .format("jdbc")
          .option("url", jdbc_url)
          .option("dbtable", f"dbo.{table_name}")
          .option("user", username)
          .option("password", password)
          .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
          .mode("append")
          .save()
    )

    print(f"{table_name} loaded successfully.")

# ==========================
# Load Gold Tables
# ==========================

write_to_sql(jobs_fact_sql_df, "jobs_fact")
write_to_sql(skill_demand_df, "skill_demand")
write_to_sql(company_hiring_df, "company_hiring")
write_to_sql(location_hiring_df, "location_hiring")
write_to_sql(salary_summary_df, "salary_summary")
write_to_sql(tech_stack_demand_df, "tech_stack_demand")

print("All Gold tables successfully loaded into Azure SQL Database.")

ConnectionSuccessful
1


jobs_fact loaded successfully.
skill_demand loaded successfully.
company_hiring loaded successfully.
location_hiring loaded successfully.
salary_summary loaded successfully.
tech_stack_demand loaded successfully.
All Gold tables successfully loaded into Azure SQL Database.
